# Step 4. Transfer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/4_transfer.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**So far.** Step 2 made the trial before a factor. Step 3 made blocks that are mostly congruent or mostly incongruent with a weight, generated one by one. This step uses both ideas one level up: the mini-block is the unit, and the whole session is one design.

**The question.** Does adapting to frequent conflict make it harder to change the task?

**Prediction if it is one control setting.** Where conflict is frequent the current task is shielded more, so a change of task costs more: the switch cost is larger in the mostly incongruent context.

**Why word reading.** The switch cost is taken on word-reading mini-blocks. Word reading has almost no congruency effect, so the proportion itself hardly touches the trials the cost is measured on.

**The analysis.** Switch cost = response time in word-reading mini-blocks that follow colour naming (switch) minus response time in word-reading mini-blocks that follow word reading (repeat). Compare it between the two contexts.

### As before, plus the task and the context

Both are regular factors. Neither changes from trial to trial: each is held for a whole mini-block.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, MinimumTrials,
                      CrossBlock, synthesize_trials, tabulate_experiments, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says
from sweetpea import Nest, IterateGen

context = Factor("context", ["mostly congruent", "mostly incongruent"])  # new: held for a whole mini-block
task = Factor("task", ["name the colour", "read the word"])              # new: held for a whole mini-block

In [ ]:
# a derived factor: it follows from colour and word on the same trial
def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word


congruency = Factor("congruency", [
    DerivedLevel("congruent", WithinTrial(same, [colour, word])),
    DerivedLevel("incongruent", WithinTrial(different, [colour, word])),
])

### New: what happens at the border between two mini-blocks?

Two transition factors, as in step 2. Here they will look back one mini-block, not one trial: does the task change, and which context came before?

In [ ]:
def task_repeats(task):
    return task[0] == task[-1]


def task_switches(task):
    return task[0] != task[-1]


task_transition = Factor("task_transition", [
    DerivedLevel("repeat", Transition(task_repeats, [task])),
    DerivedLevel("switch", Transition(task_switches, [task])),
])


def after_mostly_congruent(context):
    return context[-1] == "mostly congruent"


def after_mostly_incongruent(context):
    return context[-1] == "mostly incongruent"


context_before = Factor("context_before", [
    DerivedLevel("mostly congruent", Transition(after_mostly_congruent, [context])),
    DerivedLevel("mostly incongruent", Transition(after_mostly_incongruent, [context])),
])

### New: what the context makes frequent

In step 3 a weight on congruency set the proportion of a block. But a level has one weight, and here the proportion depends on the context, which changes from mini-block to mini-block. SweetPea has no weight that depends on another factor. So the weight sits on a helper factor that says whether a trial is the kind its context makes frequent: congruent in a mostly congruent mini-block, incongruent in a mostly incongruent one. A derived factor can follow from another derived factor: this one follows from congruency and the context.

In [ ]:
def frequent_here(congruency, context):
    return (congruency == "congruent") == (context == "mostly congruent")


def rare_here(congruency, context):
    return not frequent_here(congruency, context)


frequency = Factor("frequency", [
    DerivedLevel("frequent", WithinTrial(frequent_here, [congruency, context]), 3),
    DerivedLevel("rare", WithinTrial(rare_here, [congruency, context]), 1),
])

### New: a block inside a block

Two blocks, each with its own crossing:

- **within a mini-block**: the frequency crossed with the ink colour, 12 trials;
- **between mini-blocks**: the context, the context before, the task and the task transition. That is every kind of border, 16 mini-blocks, plus a first one that has no mini-block before it.

`Nest` puts the first inside the second: every combination of the outer block is held for one whole mini-block.

In [ ]:
mini_block = CrossBlock(design=[colour, word, congruency, frequency, context],
                        crossing=[frequency, colour],
                        constraints=[])
between_mini_blocks = CrossBlock(design=[context, context_before, task, task_transition],
                                 crossing=[context, context_before, task, task_transition],
                                 constraints=[])
block = Nest(outer_block=between_mini_blocks, inner_block=mini_block, alignment="post preamble")
experiments = synthesize_trials(block, 1, IterateGen)       # 204 trials: too long for CMSGen
after_start = list(range(12, 204))                           # without the first mini-block
tabulate_experiments(block, experiments, [context_before, context, task, task_transition], trials=after_start)

Experiment 0:
context_before mostly congruent   | context mostly congruent   | task name the colour | task_transition repeat | frequency 12 | proportion 6.25%
context_before mostly congruent   | context mostly congruent   | task name the colour | task_transition switch | frequency 12 | proportion 6.25%
context_before mostly congruent   | context mostly congruent   | task read the word   | task_transition repeat | frequency 12 | proportion 6.25%
context_before mostly congruent   | context mostly congruent   | task read the word   | task_transition switch | frequency 12 | proportion 6.25%
context_before mostly congruent   | context mostly incongruent | task name the colour | task_transition repeat | frequency 12 | proportion 6.25%
context_before mostly congruent   | context mostly incongruent | task name the colour | task_transition switch | frequency 12 | proportion 6.25%
context_before mostly congruent   | context mostly incongruent | task read the word   | task_transition repeat | frequency 12 | proportion 6.25%
context_before mostly congruent   | context mostly incongruent | task read the word   | task_transition switch | frequency 12 | proportion 6.25%
context_before mostly incongruent | context mostly congruent   | task name the colour | task_transition repeat | frequency 12 | proportion 6.25%
context_before mostly incongruent | context mostly congruent   | task name the colour | task_transition switch | frequency 12 | proportion 6.25%
context_before mostly incongruent | context mostly congruent   | task read the word   | task_transition repeat | frequency 12 | proportion 6.25%
context_before mostly incongruent | context mostly congruent   | task read the word   | task_transition switch | frequency 12 | proportion 6.25%
context_before mostly incongruent | context mostly incongruent | task name the colour | task_transition repeat | frequency 12 | proportion 6.25%
context_before mostly incongruent | context mostly incongruent | task name the colour | task_transition switch | frequency 12 | proportion 6.25%
context_before mostly incongruent | context mostly incongruent | task read the word   | task_transition repeat | frequency 12 | proportion 6.25%
context_before mostly incongruent | context mostly incongruent | task read the word   | task_transition switch | frequency 12 | proportion 6.25%

Every kind of border has 12 trials: one mini-block each. The first mini-block has nothing before it; it is the start and is left out of the counts.

### Within a mini-block

The proportion in each context, and the ink colours. Then one mini-block on its own: the second one.

In [ ]:
tabulate_experiments(block, experiments, [context, congruency], trials=after_start)
tabulate_experiments(block, experiments, [context, colour], trials=after_start)
tabulate_experiments(block, experiments, [context, task, congruency, colour], trials=list(range(12, 24)))

Experiment 0:
context mostly congruent   | congruency congruent   | frequency 72 | proportion 37.5%
context mostly congruent   | congruency incongruent | frequency 24 | proportion 12.5%
context mostly incongruent | congruency congruent   | frequency 24 | proportion 12.5%
context mostly incongruent | congruency incongruent | frequency 72 | proportion 37.5%

Experiment 0:
context mostly congruent   | colour red   | frequency 32 | proportion 16.666666666666664%
context mostly congruent   | colour green | frequency 32 | proportion 16.666666666666664%
context mostly congruent   | colour blue  | frequency 32 | proportion 16.666666666666664%
context mostly incongruent | colour red   | frequency 32 | proportion 16.666666666666664%
context mostly incongruent | colour green | frequency 32 | proportion 16.666666666666664%
context mostly incongruent | colour blue  | frequency 32 | proportion 16.666666666666664%

Experiment 0:
context mostly congruent   | task name the colour | congruency congruent   | colour red   | frequency 3 | proportion 25.0%
context mostly congruent   | task name the colour | congruency congruent   | colour green | frequency 3 | proportion 25.0%
context mostly congruent   | task name the colour | congruency congruent   | colour blue  | frequency 3 | proportion 25.0%
context mostly congruent   | task name the colour | congruency incongruent | colour red   | frequency 1 | proportion 8.333333333333332%
context mostly congruent   | task name the colour | congruency incongruent | colour green | frequency 1 | proportion 8.333333333333332%
context mostly congruent   | task name the colour | congruency incongruent | colour blue  | frequency 1 | proportion 8.333333333333332%
context mostly congruent   | task read the word   | congruency congruent   | colour red   | frequency 0 | proportion 0.0%
context mostly congruent   | task read the word   | congruency congruent   | colour green | frequency 0 | proportion 0.0%
context mostly congruent   | task read the word   | congruency congruent   | colour blue  | frequency 0 | proportion 0.0%
context mostly congruent   | task read the word   | congruency incongruent | colour red   | frequency 0 | proportion 0.0%
context mostly congruent   | task read the word   | congruency incongruent | colour green | frequency 0 | proportion 0.0%
context mostly congruent   | task read the word   | congruency incongruent | colour blue  | frequency 0 | proportion 0.0%
context mostly incongruent | task name the colour | congruency congruent   | colour red   | frequency 0 | proportion 0.0%
context mostly incongruent | task name the colour | congruency congruent   | colour green | frequency 0 | proportion 0.0%
context mostly incongruent | task name the colour | congruency congruent   | colour blue  | frequency 0 | proportion 0.0%
context mostly incongruent | task name the colour | congruency incongruent | colour red   | frequency 0 | proportion 0.0%
context mostly incongruent | task name the colour | congruency incongruent | colour green | frequency 0 | proportion 0.0%
context mostly incongruent | task name the colour | congruency incongruent | colour blue  | frequency 0 | proportion 0.0%
context mostly incongruent | task read the word   | congruency congruent   | colour red   | frequency 0 | proportion 0.0%
context mostly incongruent | task read the word   | congruency congruent   | colour green | frequency 0 | proportion 0.0%
context mostly incongruent | task read the word   | congruency congruent   | colour blue  | frequency 0 | proportion 0.0%
context mostly incongruent | task read the word   | congruency incongruent | colour red   | frequency 0 | proportion 0.0%
context mostly incongruent | task read the word   | congruency incongruent | colour green | frequency 0 | proportion 0.0%
context mostly incongruent | task read the word   | congruency incongruent | colour blue  | frequency 0 | proportion 0.0%

### The sequence as a table

One row per trial, with the mini-blocks numbered.

In [ ]:
session = pd.DataFrame(experiments[0])
session.insert(0, "block", session.index // 12 + 1)
counted = session[session["block"] > 1]
display(session.iloc[10:14])

,block,context,task,colour,word,congruency,frequency,context_before,task_transition
10,1,mostly incongruent,name the colour,blue,green,incongruent,frequent,,
11,1,mostly incongruent,name the colour,green,green,congruent,rare,,
12,2,mostly congruent,name the colour,red,green,incongruent,rare,mostly incongruent,repeat
13,2,mostly congruent,name the colour,blue,blue,congruent,frequent,mostly incongruent,repeat


### What is not in the design?

The two trials at the border themselves: the last trial of one mini-block and the first trial of the next. They are left to chance.

**Check.** Both crossings are exact, and every kind of border occurs once. What is thin? Each kind of border is one mini-block of 12 trials; more need several sequences. The two trials at the border are left to chance: is that acceptable, or should the first trial of a mini-block be pinned?

**The sampler.** `CMSGen` did not finish this design in five minutes. `IterateGen` takes a few seconds, but it does not draw at random among all possible sequences: it returns the solutions in the order the solver finds them. Look at the order of the trials before you use it.

**Try.** Take `context_before` out of the outer block. How many mini-blocks are left, and does `CMSGen` work again?

**The idea.** Each question added a factor, and each factor multiplies the design. Nesting keeps both levels counterbalanced at once. What it leaves open is still a choice, and which choice is acceptable depends on what you compare.